In [1]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [2]:
api_wrapper = WikipediaAPIWrapper(top_k_result=1 , doc_content_chars_max=200)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper)

In [3]:
!pip install sentence-transformers

from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

loader1 = PyPDFLoader("all_docs/AB-PMJAY.pdf").load()
loader2 = PyPDFLoader("all_docs/ayushman_bharat.pdf").load()
loader3 = PyPDFLoader("all_docs/NHM_more_information.pdf").load()
loader4 = PyPDFLoader("all_docs/PM-JAY.pdf").load()




[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
c:\Users\aryan\OneDrive\Desktop\NGO\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.
C:\Users\aryan\AppData\Local\Temp\ipykernel_34660\2398748590.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(


In [4]:
all_docs = loader1 + loader2 + loader3 + loader4 
documents = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
).split_documents(all_docs)

vectordb = FAISS.from_documents(documents , embedding)
retriever = vectordb.as_retriever()


In [21]:
from dotenv import load_dotenv
load_dotenv()
import os




In [6]:
from langchain_classic.tools.retriever import create_retriever_tool

retriever_tool = create_retriever_tool(
    retriever,
    "health_policy_scheme_search",
    "Use this tool to search trusted health policy and government scheme documents "
    "including Ayushman Bharat, PM-JAY, National Health Mission (NHM), and other "
    "central/state healthcare policies in India. Always use this tool when the "
    "question is related to health schemes, eligibility, benefits, documents required, "
    "or policy guidelines."
)


In [7]:
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

arxiv_wrapper=ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=200)
arxiv=ArxivQueryRun(api_wrapper=arxiv_wrapper)

In [8]:
tools = [wiki,arxiv,retriever_tool]
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\aryan\\OneDrive\\Desktop\\NGO\\venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=3, lang='en', load_all_available_meta=False, doc_content_chars_max=200)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=200)),
 StructuredTool(name='health_policy_scheme_search', description='Use this tool to search trusted health policy and government scheme documents including Ayushman Bharat, PM-JAY, National Health Mission (NHM), and other central/state healthcare policies in India. Always use this tool when the question is related to health schemes, eligibility, benefits, documents required, or poli

In [17]:
!pip install langchain-google-genai
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.tools import tool

from langchain_groq import ChatGroq
import os

llm = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name="openai/gpt-oss-20b",
    temperature=0.2
)



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an AI Health Policy & Scheme Navigator Assistant for India. "
     "You must provide accurate, reliable, and simple explanations about health schemes "
     "such as Ayushman Bharat (PM-JAY), National Health Mission, state schemes, and WHO policy guidance. "
     "Always use the retriever tool to fetch information from official PDFs before answering. "
     "If the user asks about eligibility, benefits, coverage, documents required, or implementation details, "
     "search the documents and then respond. "
     "Explain in simple, human-understandable language. "
     "Do not hallucinate. If information is not found, say you are unsure."
    ),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])


In [19]:
agent = create_tool_calling_agent(llm , tools , prompt)
## Agent Executer
agent_executor=AgentExecutor(agent=agent,tools=tools,verbose=True)
agent_executor



AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: message_formatter(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['input'], optional_variables=['agent_scratchpad'], input_types={'agent_scratchpad': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')

In [20]:
result = agent_executor.invoke({"input":"India's major health schemes and policies"})
print(result)




> Entering new AgentExecutor chain...

Invoking: `health_policy_scheme_search` with `{'query': 'Ayushman Bharat PM-JAY official PDF'}`


References:  
 
• https://www.nhp.gov.in/pm-ayushman-bharat-health-infrastructure-mission_pg 
• https://www.nhp.gov.in/ayushman-bharat-yojana_pg 
• https://abdm.gov.in/home/abdm 
• https://ab-hwc.nhp.gov.in/download/document/HWCs_Booklet_English_updated_5_April_2022.pdf  
• https://pib.gov.in/PressReleasePage.aspx?PRID=1799064 
• https://pib.gov.in/PressReleaseIframePage.aspx?PRID=1696433 
• https://pib.gov.in/PressReleasePage.aspx?PRID=1761175 
• https://pib.gov.in/PressReleasePage.aspx?PRID=1816131 
• https://pib.gov.in/PressReleasePage.aspx?PRID=1758502 
• https://www.pib.gov.in/PressReleasePage.aspx?PRID=1738169 
• https://pib.gov.in/PressReleasePage.aspx?PRID=1766289 
• https://www.india.gov.in/spotlight/ayushman-bharat-national-health-protection-mission 
• https://nha.gov.in/PM-JAY 
• https://nha.gov.in/img/resources/Annual-Report-2020-21.pdf 